# Combined Measurement Model

## Review

In the previous two sections, we modelled $O_k$ and $C_k$ which are the matrices containing potentially true observations and false positives respectively.

### $O_k$ - True Detections

The number of true detections in the single object case is 0 or one and therefore Bernoulli distributed:

$$
p(O_k) = \begin{cases}
1 - P_D(x_k) & \text{if } O_k = \{\} \\
P_D(x_k) & \text{if } O_k = o_k
\end{cases}
$$

From which, in combination with sensor model, we get a combined measurement model for true detections

$$
P(O_k \mid x_k) =
\begin{cases}
P_D(x_k)g(o_k \mid x_k) & \text{if } O_k = o_k \\
1 - P_D(x_k) & \text{if } O_k = \{\}
\end{cases}
$$

### $C_k$ - Clutter Detections

Modelling the clutter detection matrix $C_k$ comes in two parts:
- The number of clutter detections, $N_k$, that follows a Poisson distribution $P(N_k) \sim \text{Po}(\bar{\lambda}_c)$
- and the i.i.d. measurements $c_k^1 ... c_k^{N_k}$ that are drawn from $c_k^i \sim f_c(c_k^i)$

Together result in the model:

$$
P(C_k, N_k) = \text{Po}(\bar{\lambda}_c) \prod_{i = 1}^{N_k} f_c(c_k^i)
$$

## New Objective: Complete Measurement Model For a Single Object in Clutter

Now we wish to arrive at a combined measurement model for the measurement matrix

$$Z_k = \Pi(O_k, C_k)$$

which is the matrix of all measurements, including both true and false positives. The model we seek, specifically, is

$$p(Z_k \mid x_k)$$

### Complicating Factors

The challenge is that we do not know
1. The size of $Z_k$ indicating the total number of detections clutter or otherwise is random
1. which, if any, i.i.d detections in $Z_k$ is an object detection, is unknown and random.

If we could condition on this information, we could put together a model of $p(Z_k \mid x_k)$ from what we already know, covered above. Unfortunately, we are not given such clarifying information, so we must account for it in our probability model.

### Introducing $\theta$ and $m$

$m$ Represents the unknown total number of measurements in $Z_k$, both any potential object detection and all false positives.

$$m = |Z_k|$$

And $\theta$ is an indicator of which measurement, if any, corresponds to an object detection.

<br>
$$
\theta =
\begin{cases}
0 & \text{if no detections} \\
i \in \{1 ... m\} & \text{if } z^i \text{ is the object measurement}
\end{cases}
$$
<br>

The general trick here is to specify variables that, if known, would simplify our model. For example, if we knew $theta=3$, then we'd know

<br>
$$P(Z_k \mid \theta = 3, x) = P_d(x_k)g(z^{\theta = 3} \mid x_k)\prod_{i \not = 3}f_c(z^i_k)$$
<br>

and our problem would be solved. But $\theta$ is unknown. So what do we do?

### Using $m$ and $theta$

First off, given a measurement matrix $Z$, we get it's width $m$ for free, making it true that 

<br>
$$P(Z \mid x) = P(Z, m \mid x)$$
<br>

However, $\theta$ remains unknown, and so our probability $P(Z, m | x)$ must considering all possible values of $\theta$, by the law of total probability:

<br>
\begin{align}
P(Z, m \mid x) &= \sum\limits_{\theta=0}^m P(Z, \theta, m \mid x) \\[2ex]
&= \sum\limits_{\theta=1}^m P(Z \mid \theta, m, x)P(\theta, m \mid x)
\end{align}

#### First Term: $P(Z \mid \theta, m, x)$

To compute the first term, we leverage the fact that $\theta$ and are given conditions and can write:

<br>
$$
P(Z_k \mid \theta, m, x) = 
\begin{cases}
g(z^{\theta} \mid x_k)\prod\limits_{i \not = \theta}^m f_c(z^i_k) & \text{if } \theta > 0 \\
\prod\limits_{i = 1}^m f_c(z^i_k) & \text{if } \theta = 0
\end{cases}
$$

<br>
And are left to determine 
<br>

$$P(\theta, m \mid x) = \text{???}$$

#### Second Term: $P(\theta, m \mid x)$

$Z = \prod(O_k, C_k)$ which is a random shuffling of clutter and true detections (if any true detections exist). 

Granted that we know detection occurs and that there are $m - 1$ clutter detections, $\theta$ should have $1/m$ probability for all values, because the true object detection could be any column of $Z_k$. However, detections aren't granted, and so we must consider their probabilities in our formulation. 

1. We recall that the probability of a detection is Bernoulli distributed with probability parameter $P_D(x)$.
2. We recall that the number of clutter distributions is Poisson distributed with parameter $\bar{\lambda}_c$.

<br>
$$
P(\theta \mid x) = 
\begin{cases}
P_D(x_k) \text{Po}(1 - m; \bar{\lambda}_c) \frac{1}{m} & \text{if } \theta > 0 \\[2ex]
(1 - P_D(x)) \text{Po}(m; \bar{\lambda}_c) & \text{if } \theta = 0
\end{cases}
$$

### Combined Model

Bringing the two parts together under the summation, we get the following:

\begin{align}
P(Z | x) &= \sum_{\theta=0}^m P(Z, \theta, m \mid x) \\[2ex]
&= (1 - P_D(x)) \text{Po}(m; \bar{\lambda}_c) \prod\limits_{i=1}^m f_c(z^i_k) + \sum_{\theta=1}^m P(Z \mid \theta, m, x)P(\theta, m \mid x) \\[2ex]
&= (1 - P_D(x)) \text{Po}(m; \bar{\lambda}_c) \prod\limits_{i=1}^m f_c(z^i_k) + P_D(x_k) \text{Po}(m - 1; \bar{\lambda}_c) \sum\limits_{\theta=1}^m g(z^{\theta} | x_k)  \prod\limits_{i \not = \theta}^m f_c(z^i_k) \frac{1}{m} \\[2ex]
\end{align}

Things to keep in mind:

1. We know the form of the Poisson PMF, so I left it out for brevity.
2. The rate of clutter detections may vary over the field of view $V$. $\bar{\lambda}_c$ is the average rate of clutter detections over the entire field of view $V$:
   $$
   \bar{\lambda}_c = \int_V \lambda_c(c) dc
   $$
3. $f_c(c)$ is the spatial probability density function for clutter.
   $$
   f_c(c) = \frac{\lambda_c(c)}{\bar{\lambda}_c}
   $$

For implementation purposes, plugging these terms into the above, with some algebraic reduction, will yield the final formula. 

\begin{align}
P(Z, m | x) &= \sum_{\theta=1}^m P(Z, \theta, m \mid x) \\
&= \sum_{\theta=1}^m P(Z \mid \theta, m, x)P(\theta, m \mid x)
&=
\begin{cases}
\frac{1}{m}g(z^{\theta} | x_k)\prod_{i \not = \theta}f_c(z^i_k) & \text{if } \theta > 0 \\
\prod_{i = 1}^m f_c(z^i_k) & \text{if } \theta = 0
\end{cases}
\end{align}

$$P(Z_k | \theta, x) = (1 - P_d(x_k)) \prod_{i  =1}^mf_c(z^i_k)$$